In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Import Libraries

In [2]:
import pandas as pd
import numpy as np
import string

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# EDA

In [3]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

print(train.shape)
print(test.shape)

(2000, 8)
(500, 7)


In [4]:
train.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


# Q1

In [5]:
train['answer'].value_counts()

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

In [6]:
train.isnull().sum()

id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64

# Q2

In [7]:
def clean_text(text):
    text = text.lower()
    for p in string.punctuation:
        text = text.replace(p, '')
    return text

all_words = set()

for text in train['prompt']:
    all_words.update(clean_text(str(text)).split())

len(all_words)

859

# Q3

In [8]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

row = train.loc[train['id']==1].iloc[0]

words = clean_text(row['prompt']).split()

filtered = []

for w in words:
    if w not in ENGLISH_STOP_WORDS:
        filtered.append(w)
 

len(filtered)

13

# Q4

In [9]:
combined_text = []

for _, row in train.iterrows():
    text = (
        str(row['prompt']) + " " +
        str(row['A']) + " " +
        str(row['B']) + " " +
        str(row['C']) + " " +
        str(row['D']) + " " +
        str(row['E'])
    )

    combined_text.append(text)

vectorizer = TfidfVectorizer(stop_words='english')

X = vectorizer.fit_transform(combined_text)

print(len(vectorizer.vocabulary_))

2762


# Q5

In [10]:
row = train.loc[train['id']==1].iloc[0]

prompt_vec = vectorizer.transform([row['prompt']])
a_vec = vectorizer.transform([row['A']])

cosine_similarity(prompt_vec, a_vec)[0][0]

np.float64(0.27202429519891635)

# Q6

In [11]:
correct = 0

for _, row in train.iterrows():

    prompt_vec = vectorizer.transform([row['prompt']])

    scores = []

    for choice in ['A','B','C','D','E']:
        option_vec = vectorizer.transform([row[choice]])
        sim = cosine_similarity(prompt_vec, option_vec)[0][0]
        scores.append((choice, sim))

    pred = max(scores, key=lambda x:x[1])[0]

    if pred == row['answer']:
        correct += 1

accuracy = correct / len(train)
accuracy

0.1355

# Evaluation Metric

In [12]:
def map3(actuals, predictions):
    score = 0

    for actual, pred in zip(actuals, predictions):
        if actual == pred[0]:
            score += 1
        elif actual == pred[1]:
            score += 1/2
        elif actual == pred[2]:
            score += 1/3

    return score / len(actuals)

# Q7 : 1
# Q8 : 0.5

# Q9

In [13]:
predictions = [['B','C','A']] * len(train)

majority_map3 = map3(train['answer'].tolist(),predictions)

print(majority_map3)

0.4212500000000017


# Cross-Val Split

In [14]:
train_df, val_df = train_test_split(train,test_size=0.2,stratify=train["answer"],random_state=4524)

# Model 1(Scratch): TfidfVectorizer

In [15]:
all_text = []

for col in ['prompt', 'A', 'B', 'C', 'D', 'E']:
    all_text.extend(train[col].fillna('').astype(str))

vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=50000
)

vectorizer.fit(all_text)

TfidfVectorizer(max_features=50000, stop_words='english')

In [16]:
choices = ['A', 'B', 'C', 'D', 'E']

def predict_top3(df, vectorizer):
    predictions = []

    for _, row in df.iterrows():
        prompt_vec = vectorizer.transform([str(row['prompt'])])
        scores = []

        for choice in choices:
            option_vec = vectorizer.transform([str(row[choice])])
            sim = cosine_similarity(prompt_vec,option_vec)[0][0]
            scores.append((choice, sim))

        scores.sort(key=lambda x: x[1], reverse=True)
        predictions.append([x[0] for x in scores[:3]])

    return predictions

train_predictions = predict_top3(train, vectorizer)

# Q10

In [17]:
actual = train['answer'].tolist()

score = map3(actual, train_predictions)

print("MAP@3:", score)

MAP@3: 0.31191666666666656


In [18]:
test_predictions = predict_top3(test, vectorizer)

test_predictions = [" ".join(pred) for pred in test_predictions]

# Submission Cell

In [19]:
submission = pd.DataFrame({
    "ID": test["id"],
    "Prediction": test_predictions
})

submission.to_csv("submission.csv", index=False)

submission.head()

,ID,Prediction
0,1,A B C
1,2,A B C
2,3,A D C
3,4,A E C
4,5,A C E
